# 🎨 축구 피치 시각화 마스터: mplsoccer 가이드

`mplsoccer`는 파이썬에서 축구 피치 및 이벤트 데이터를 시각화하는 데 가장 널리 쓰이는 표준 라이브러리입니다.

### 다루는 내용:
1. **기본 피치 및 커스텀 스타일링** (가로, 세로, 잔디색, 다크모드)
2. **패스 맵 (Pass Map)**: 성공/실패 패스 화살표 시각화
3. **히트맵 (Heatmap) & KDE(커널 밀도 추정)**: 선수의 활동 반경 및 히트존
4. **수비 액션 맵 (Defensive Actions)**: 태클, 인터셉트 위치 시각화

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsbombpy import sb
from mplsoccer import Pitch, VerticalPitch

# 2022 월드컵 결승전 이벤트 데이터 로드
events = sb.events(match_id=3869685)

## 1. 패스 맵 (Pass Map): 메시 vs 음바페 패스 비교

In [ ]:
def draw_player_pass_map(events_df, player_name, ax, pitch, title):
    passes = events_df[(events_df['type'] == 'Pass') & (events_df['player'] == player_name)].copy()
    passes['x'] = passes['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
    passes['y'] = passes['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else np.nan)
    passes['end_x'] = passes['pass_end_location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
    passes['end_y'] = passes['pass_end_location'].apply(lambda loc: loc[1] if isinstance(loc, list) else np.nan)
    
    completed = passes[passes['pass_outcome'].isna()]
    incomplete = passes[passes['pass_outcome'].notna()]
    
    pitch.arrows(completed['x'], completed['y'], completed['end_x'], completed['end_y'],
                 color='#00ff85', width=2, headwidth=4, headlength=4, label=f'성공 ({len(completed)})', ax=ax, alpha=0.7)
    pitch.arrows(incomplete['x'], incomplete['y'], incomplete['end_x'], incomplete['end_y'],
                 color='#ff4b4b', width=1.5, headwidth=3, headlength=3, label=f'실패 ({len(incomplete)})', ax=ax, alpha=0.5)
    
    ax.set_title(f"{player_name} ({title})", color='white', fontsize=14, pad=12)
    ax.legend(facecolor='#1e1e1e', edgecolor='white', labelcolor='white', loc='lower center', ncol=2)

# 1행 2열 피치 생성
pitch = Pitch(pitch_type='statsbomb', pitch_color='#1e1e1e', line_color='#7c7c7c')
fig, axs = pitch.grid(nrows=1, ncols=2, figheight=8, space=0.1, title_height=0.08, grid_height=0.82)
fig.set_facecolor('#1e1e1e')

draw_player_pass_map(events, 'Lionel Andrés Messi Cuccittini', axs['pitch'][0], pitch, '아르헨티나')
draw_player_pass_map(events, 'Kylian Mbappé Lottin', axs['pitch'][1], pitch, '프랑스')

axs['title'].set_facecolor('#1e1e1e')
axs['title'].text(0.5, 0.5, '2022 World Cup Final: Pass Map Comparison', color='white',
                 fontsize=20, ha='center', va='center', weight='bold')
plt.show()

## 2. 커널 밀도 추정 히트맵 (KDE Heatmap)

In [ ]:
# 메시의 모든 볼 터치/이벤트 위치
messi_events = events[events['player'] == 'Lionel Andrés Messi Cuccittini'].copy()
messi_events['x'] = messi_events['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
messi_events['y'] = messi_events['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else np.nan)
messi_events = messi_events.dropna(subset=['x', 'y'])

pitch = Pitch(pitch_type='statsbomb', line_zorder=2, pitch_color='#181818', line_color='#c7d5cc')
fig, ax = pitch.draw(figsize=(10, 7))
fig.set_facecolor('#181818')

# KDE Plot
kde = pitch.kdeplot(
    messi_events['x'], messi_events['y'], ax=ax,
    fill=True, levels=100, thresh=0.05,
    cmap='hot', alpha=0.7
)

ax.set_title("Lionel Messi - Heatmap / Activity Zone (2022 WC Final)", color='white', fontsize=16, pad=15, weight='bold')
plt.show()